# 01 — MSO Foundations

This lab turns the module vocabulary into real, low-token Claude API calls. It covers context budgets, non-determinism, prompting modes, SDK access, and streaming. Start with the [module index](../course%20content%20HTML/01-mso-foundations/index.html).

> Running all cells requires `OPENROUTER_API_KEY` and uses paid API tokens.

## Connect the Anthropic SDK through OpenRouter

The Anthropic SDK provides the Messages API surface. The shared client sends those requests to OpenRouter, while the `anthropic/` model prefix selects Anthropic as the provider.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_support.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

The helper loads the root `.env`, requires one OpenRouter API key, and constructs `Anthropic(base_url="https://openrouter.ai/api")`. The key is never printed.

In [2]:
from study_support import claude_client, claude_model, message_text

client = claude_client()
MODEL = claude_model()
MODEL

'anthropic/claude-sonnet-5'

## Tokens and context are a fixed budget

The context window holds instructions, user content, tool definitions, prior turns, and reserved output together. `max_tokens` limits generated output; it does not make input free. [Course screen 2](../course%20content%20HTML/01-mso-foundations/02-how-llms-behave.html#screen-2--how-llms-behave-tokens-context-sampling-non-determinism)

In [3]:
system_prompt = "Classify support tickets. Answer with a label and one short reason."
messages = [{
    "role": "user",
    "content": "I was charged twice for the same monthly plan.",
}]

A response includes token usage. Inspect it instead of estimating production cost from character counts.

In [4]:
response = client.messages.create(
    model=MODEL,
    max_tokens=80,
    system=system_prompt,
    messages=messages,
)
print(message_text(response))
print(response.usage)

**Label:** Billing Issue – Duplicate Charge

**Reason:** The customer reports being billed twice for a single subscription cycle, indicating a payment processing error that requ
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=42, output_tokens=80, output_tokens_details=OutputTokensDetails(thinking_tokens=22), server_tool_use=None, service_tier='standard', speed='standard', cost=0.000884, is_byok=False, cost_details={'upstream_inference_cost': 0.000884, 'upstream_inference_prompt_cost': 8.4e-05, 'upstream_inference_completions_cost': 0.0008})


## Sampling makes outputs non-deterministic

The same request can produce different valid wording. Tests should grade the requirement, not a single golden sentence. Two runs may still happen to match. [Course explanation](../course%20content%20HTML/01-mso-foundations/02-how-llms-behave.html#sampling-why-the-same-prompt-can-give-different-answers)

In [5]:
def ask_once():
    result = client.messages.create(
        model=MODEL,
        max_tokens=40,
        messages=[{"role": "user", "content": "Name one benefit of API streaming in one sentence."}],
    )
    return message_text(result)

Compare two independent samples. Do not assert that their exact text differs.

In [6]:
samples = [ask_once() for _ in range(2)]
for sample in samples:
    print(f"- {sample}")

- API streaming allows clients to receive and process data incrementally as it becomes available, reducing latency and improving the user experience for large or
- API streaming lets applications receive and process data incrementally in real time, reducing latency and improving responsiveness compared to waiting


## Examples trade tokens for control

Zero-shot is shortest. One-shot demonstrates a target shape. Multi-shot is useful when several examples teach a boundary that prose alone does not. [Course screen 4](../course%20content%20HTML/01-mso-foundations/04-prompting-modes.html#screen-4--prompting-modes-zero-shot-one-shot-multi-shot)

In [7]:
few_shot_messages = [
    {"role": "user", "content": "The parcel arrived cracked."},
    {"role": "assistant", "content": "damage | replace"},
    {"role": "user", "content": "My invoice lists an item I never ordered."},
]

The example spends context tokens, but it gives Claude a concrete output contract.

In [8]:
classified = client.messages.create(
    model=MODEL,
    max_tokens=20,
    system="Return exactly: category | next_action",
    messages=few_shot_messages,
)
print(message_text(classified))

billing_error | investigate




## Streaming changes delivery, not the task

A stream improves time-to-first-token. Accumulate complete text before committing it to durable conversation state. [Course screen 5](../course%20content%20HTML/01-mso-foundations/05-technical-substrate.html#synchronous-streaming-and-real-time-responses)

In [9]:
parts = []
with client.messages.stream(
    model=MODEL, max_tokens=60,
    messages=[{"role": "user", "content": "Explain context windows in two short sentences."}],
) as stream:
    for text in stream.text_stream:
        parts.append(text)
        print(text, end="", flush=True)
complete_text = "".join(parts)

A context window is the amount of text (measured in tokens) that an AI model can "see" and process at one time when generating a response. Anything beyond that limit gets truncated or forgotten, which is why

## Try it

Change the ticket and compare zero-shot with the one-shot contract. Record the returned usage each time. Then explain which change affected quality and which affected token cost.